# MILK10k — Step 4: When is dermoscopy needed? (acquisition policy)
Uses only the out-of-fold probabilities saved by notebook 03 (model **late_drop**), so it runs on CPU in minutes.

For each lesion the model first sees **clinical photo + metadata (CM)**. A policy ranks lesions by how much dermoscopy is expected to help; the top *q%* get dermoscopy and their prediction is replaced by the **all-inputs (CDM)** prediction. We sweep *q* from 0 to 100% and compare policies.

**Actionable** = needs treatment or excision: BCC, MEL, SCCKA, MAL_OTH (malignant) + AKIEC (actinic keratosis / intraepidermal carcinoma).

In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.ensemble import HistGradientBoostingRegressor

def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None

NPZ = find_file("oof_probs_fusion.npz");  assert NPZ, "attach notebook-03 output (Add Input > Your Work > Notebooks)"
FOLDS_CSV = find_file("milk10k_lesions_folds.csv"); assert FOLDS_CSV, "attach notebook-02 output too"
OUT = "/kaggle/working/stage4"; os.makedirs(OUT, exist_ok=True)
print("probs:", NPZ, "\nfolds:", FOLDS_CSV)

les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]
act_idx = [CLASSES.index(c) for c in ACTIONABLE]
y = les["y"].values.astype(int); fold = les["fold"].values
y_act = np.isin(y, act_idx).astype(int)

npz = np.load(NPZ)
CFG = "late_drop"
P_CM, P_CDM = npz[f"{CFG}|CM"].astype(float), npz[f"{CFG}|CDM"].astype(float)
assert np.allclose(P_CM.sum(1), 1, atol=1e-3) and np.allclose(P_CDM.sum(1), 1, atol=1e-3)
print("lesions:", len(y), "| actionable:", y_act.sum(), f"({y_act.mean():.1%})")

## Metrics for a mixed prediction set

In [ ]:
def metrics(P):
    pred = P.argmax(1); s_act = P[:, act_idx].sum(1)
    return {"macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
            "bal_acc": balanced_accuracy_score(y, pred),
            "act_auc": roc_auc_score(y_act, s_act),
            "act_sens": ((s_act >= 0.5) & (y_act == 1)).sum() / y_act.sum(),     # sensitivity for actionable lesions
            "act_spec": ((s_act < 0.5) & (y_act == 0)).sum() / (y_act == 0).sum(),
            "MEL_recall": recall_score(y == CLASSES.index("MEL"), pred == CLASSES.index("MEL")),
            "SCCKA_recall": recall_score(y == CLASSES.index("SCCKA"), pred == CLASSES.index("SCCKA"))}

lo, hi = metrics(P_CM), metrics(P_CDM)
print(pd.DataFrame({"CM only (0% dermoscopy)": lo, "CDM for all (100%)": hi}).round(3))

## Policies
All scores use **only information available before dermoscopy** (the CM prediction + metadata), except the oracle.
* `learned_vod` — "value of dermoscopy": a gradient-boosting regressor trained on the **other 4 folds** to predict the gain in log-probability of the true class, log P_CDM(y) − log P_CM(y). Its inputs are CM probabilities, their entropy/margin, and metadata. Cross-fitted, so no lesion is scored by a model that saw its label.
* `oracle` uses the true labels (ceiling, not deployable).

In [ ]:
eps = 1e-9
def entropy(P): return -(P * np.log(P + eps)).sum(1)
s_act_CM = P_CM[:, act_idx].sum(1)
top2 = np.sort(P_CM, 1)[:, -2:]

scores = {
    "random":        np.random.RandomState(0).rand(len(y)),
    "entropy":       entropy(P_CM),
    "least_conf":    1 - P_CM.max(1),
    "small_margin":  -(top2[:, 1] - top2[:, 0]),
    "near_boundary": -np.abs(s_act_CM - 0.5),                 # close to treat / don't-treat boundary
}

# learned value-of-dermoscopy (cross-fitted over folds)
site_d = pd.get_dummies(les["site"].fillna("unknown")).values.astype(float)
feat = np.column_stack([P_CM, entropy(P_CM), top2[:, 1] - top2[:, 0], s_act_CM,
                        les["age_approx"].fillna(-1).values, les["skin_tone_class"].values,
                        (les["sex"] == "male").values, site_d])
gain = np.log(P_CDM[np.arange(len(y)), y] + eps) - np.log(P_CM[np.arange(len(y)), y] + eps)
vod = np.zeros(len(y))
for f in range(5):
    tr, te = fold != f, fold == f
    reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                        l2_regularization=1.0, random_state=0).fit(feat[tr], gain[tr])
    vod[te] = reg.predict(feat[te])
scores["learned_vod"] = vod
scores["oracle"] = gain
print("learned VoD vs true gain, Spearman:", round(pd.Series(vod).corr(pd.Series(gain), method="spearman"), 3))

## Sweep the dermoscopy rate

In [ ]:
RATES = np.round(np.arange(0, 1.0001, 0.05), 2)
def mixed(score, q):
    n_req = int(round(q * len(y)))
    req = np.zeros(len(y), bool)
    if n_req: req[np.argsort(-score)[:n_req]] = True
    return np.where(req[:, None], P_CDM, P_CM), req

rows = []
for name, s in scores.items():
    for q in RATES:
        P, _ = mixed(s, q)
        rows.append(dict(policy=name, rate=q, **metrics(P)))
curves = pd.DataFrame(rows); curves.to_csv(f"{OUT}/acquisition_curves.csv", index=False)

# headline: smallest dermoscopy rate reaching 95% of the CM->CDM improvement
def rate_needed(metric, frac=0.95):
    target = lo[metric] + frac * (hi[metric] - lo[metric])
    out = {}
    for name, d in curves.groupby("policy"):
        ok = d[d[metric] >= target]
        out[name] = ok["rate"].min() if len(ok) else np.nan
    return pd.Series(out, name=f"rate for 95% of {metric} gain")
headline = pd.concat([rate_needed("macro_f1"), rate_needed("act_auc"), rate_needed("bal_acc")], axis=1)
headline.to_csv(f"{OUT}/headline_rate_needed.csv"); headline.sort_values(headline.columns[0])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
style = {"random": ("gray", "--"), "oracle": ("black", ":"), "learned_vod": ("C3", "-")}
for m, a in zip(["macro_f1", "act_auc", "act_sens"], ax):
    for name, d in curves.groupby("policy"):
        col, ls = style.get(name, (None, "-"))
        a.plot(d["rate"] * 100, d[m], label=name, color=col, ls=ls, lw=2.2 if name == "learned_vod" else 1.3)
    a.axhline(hi[m], color="C2", lw=0.8); a.set_xlabel("% of lesions sent for dermoscopy"); a.set_title(m)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/fig_acquisition_curves.png", dpi=200); plt.show()

## Per-fold stability of the headline (mean ± sd over folds)

In [ ]:
def fold_curve(score, f, metric):
    te = fold == f; vals = []
    for q in RATES:
        n_req = int(round(q * te.sum())); idx = np.where(te)[0]
        req = np.zeros(te.sum(), bool)
        if n_req: req[np.argsort(-score[te])[:n_req]] = True
        P = np.where(req[:, None], P_CDM[te], P_CM[te])
        yy = y[te]
        vals.append(f1_score(yy, P.argmax(1), average="macro", labels=range(K), zero_division=0) if metric == "macro_f1"
                    else roc_auc_score(np.isin(yy, act_idx), P[:, act_idx].sum(1)))
    return np.array(vals)

rows = []
for name in ["random", "entropy", "near_boundary", "learned_vod", "oracle"]:
    for f in range(5):
        for m in ["macro_f1", "act_auc"]:
            v = fold_curve(scores[name], f, m); target = v[0] + 0.95 * (v[-1] - v[0])
            rows.append(dict(policy=name, fold=f, metric=m, rate_needed=RATES[np.argmax(v >= target)]))
stab = pd.DataFrame(rows).groupby(["policy", "metric"])["rate_needed"].agg(["mean", "std"]).round(3).unstack()
stab.to_csv(f"{OUT}/rate_needed_per_fold.csv"); stab

## Who gets sent for dermoscopy? (at 30% rate, best non-oracle policy)

In [ ]:
best = headline.drop(index=["oracle", "random"]).iloc[:, 0].idxmin()
_, req = mixed(scores[best], 0.30)
by_class = pd.DataFrame({"n": pd.Series(y).map(dict(enumerate(CLASSES))).value_counts(),
                         "sent_for_dermoscopy": pd.Series(y[req]).map(dict(enumerate(CLASSES))).value_counts()}).fillna(0)
by_class["rate"] = (by_class["sent_for_dermoscopy"] / by_class["n"]).round(3)
print("policy:", best)
by_class.sort_values("rate", ascending=False)